In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load the dataset
train = pd.read_csv("../data/train.csv")

In [3]:
print("Train Dataset")
train.head()

Train Dataset


,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders
0,1379560,1,55,1885,136.83,152.29,0,0,177
1,1466964,1,55,1993,136.83,135.83,0,0,270
2,1346989,1,55,2539,134.86,135.86,0,0,189
3,1338232,1,55,2139,339.50,437.53,0,0,54
4,1448490,1,55,2631,243.50,242.50,0,0,40


In [4]:
test = pd.read_csv("../data/test.csv")

In [5]:
print("Test Dataset")
test.head()

Test Dataset


,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured
0,1028232,146,55,1885,158.11,159.11,0,0
1,1127204,146,55,1993,160.11,159.11,0,0
2,1212707,146,55,2539,157.14,159.14,0,0
3,1082698,146,55,2631,162.02,162.02,0,0
4,1400926,146,55,1248,163.93,163.93,0,0


In [6]:
meal_info = pd.read_csv("../data/meal_info.csv")

In [7]:
print("Meal Info Dataset")
meal_info.head()

Meal Info Dataset


,meal_id,category,cuisine
0,1885,Beverages,Thai
1,1993,Beverages,Thai
2,2539,Beverages,Thai
3,1248,Beverages,Indian
4,2631,Beverages,Indian


In [8]:
fulfillment_center = pd.read_csv("../data/fulfilment_center_info.csv")

In [9]:
print("Fulfillment Center Dataset")
fulfillment_center.head()

Fulfillment Center Dataset


,center_id,city_code,region_code,center_type,op_area
0,11,679,56,TYPE_A,3.7
1,13,590,56,TYPE_B,6.7
2,124,590,56,TYPE_C,4.0
3,66,648,34,TYPE_A,4.1
4,94,632,34,TYPE_C,3.6


In [10]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Meal info shape:", meal_info.shape)
print("Fulfillment center shape:", fulfillment_center.shape)

Train shape: (456548, 9)
Test shape: (32573, 8)
Meal info shape: (51, 3)
Fulfillment center shape: (77, 5)


In [11]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 456548 entries, 0 to 456547
Data columns (total 9 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   id                     456548 non-null  int64  
 1   week                   456548 non-null  int64  
 2   center_id              456548 non-null  int64  
 3   meal_id                456548 non-null  int64  
 4   checkout_price         456548 non-null  float64
 5   base_price             456548 non-null  float64
 6   emailer_for_promotion  456548 non-null  int64  
 7   homepage_featured      456548 non-null  int64  
 8   num_orders             456548 non-null  int64  
dtypes: float64(2), int64(7)
memory usage: 31.3 MB


In [12]:
# Merge train with meal_info
train = pd.merge(train, meal_info, on="meal_id", how="left")

# Merge train with fulfillment center info
train = pd.merge(train, fulfillment_center, on="center_id", how="left")

# Display merged dataset
train.head()

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0


In [13]:
train.info()
train.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 456548 entries, 0 to 456547
Data columns (total 15 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   id                     456548 non-null  int64  
 1   week                   456548 non-null  int64  
 2   center_id              456548 non-null  int64  
 3   meal_id                456548 non-null  int64  
 4   checkout_price         456548 non-null  float64
 5   base_price             456548 non-null  float64
 6   emailer_for_promotion  456548 non-null  int64  
 7   homepage_featured      456548 non-null  int64  
 8   num_orders             456548 non-null  int64  
 9   category               456548 non-null  object 
 10  cuisine                456548 non-null  object 
 11  city_code              456548 non-null  int64  
 12  region_code            456548 non-null  int64  
 13  center_type            456548 non-null  object 
 14  op_area                456548 non-nu

id                       0
week                     0
center_id                0
meal_id                  0
checkout_price           0
base_price               0
emailer_for_promotion    0
homepage_featured        0
num_orders               0
category                 0
cuisine                  0
city_code                0
region_code              0
center_type              0
op_area                  0
dtype: int64

In [14]:
# Sort data for proper time-series order
train = train.sort_values(["center_id", "meal_id", "week"])

In [15]:
# Lag features (previous demand values)

train["lag_1"] = train.groupby(["center_id", "meal_id"])["num_orders"].shift(1)
train["lag_2"] = train.groupby(["center_id", "meal_id"])["num_orders"].shift(2)
train["lag_3"] = train.groupby(["center_id", "meal_id"])["num_orders"].shift(3)
train["lag_4"] = train.groupby(["center_id", "meal_id"])["num_orders"].shift(4)

In [16]:
# Rolling averages (short-term demand trends)

train["rolling_mean_4"] = (
    train.groupby(["center_id", "meal_id"])["num_orders"]
         .shift(1)
         .rolling(4)
         .mean()
)

train["rolling_mean_8"] = (
    train.groupby(["center_id", "meal_id"])["num_orders"]
         .shift(1)
         .rolling(8)
         .mean()
)

In [17]:
# Discount percentage feature
train["discount"] = (train["base_price"] - train["checkout_price"]) / train["base_price"]

In [18]:
train = train.dropna()

In [19]:
print("Feature Engineering Complete")
print("Remaining rows:", len(train))

train.head()

Feature Engineering Complete
Remaining rows: 427851


,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,...,region_code,center_type,op_area,lag_1,lag_2,lag_3,lag_4,rolling_mean_4,rolling_mean_8,discount
25205,1083552,9,10,1062,162.05,182.39,0,0,1282,Beverages,...,56,TYPE_B,6.3,1149.0,1513.0,1094.0,958.0,1178.50,1051.750,0.111519
28068,1325645,10,10,1062,161.05,181.39,0,0,1473,Beverages,...,56,TYPE_B,6.3,1282.0,1149.0,1513.0,1094.0,1259.50,1103.875,0.112134
30928,1347994,11,10,1062,163.02,181.39,0,0,1363,Beverages,...,56,TYPE_B,6.3,1473.0,1282.0,1149.0,1513.0,1354.25,1190.250,0.101273
33768,1465124,12,10,1062,184.36,184.36,0,0,1295,Beverages,...,56,TYPE_B,6.3,1363.0,1473.0,1282.0,1149.0,1316.75,1254.250,0.000000
36595,1056012,13,10,1062,181.39,181.39,0,0,1054,Beverages,...,56,TYPE_B,6.3,1295.0,1363.0,1473.0,1282.0,1353.25,1265.875,0.000000


In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

train["category"] = le.fit_transform(train["category"])
train["cuisine"] = le.fit_transform(train["cuisine"])
train["center_type"] = le.fit_transform(train["center_type"])

In [21]:
# Define target variable
y = train["num_orders"]

# Define feature variables
X = train.drop(["num_orders", "id"], axis=1)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (427851, 20)
Target shape: (427851,)


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Validation set:", X_valid.shape)

Training set: (342280, 20)
Validation set: (85571, 20)


In [23]:
!pip install catboost lightgbm

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [25]:
# Target variable
y = train["num_orders"]

# Features
X = train.drop(["num_orders", "id"], axis=1)

print(X.shape)
print(y.shape)

(427851, 20)
(427851,)


In [26]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Validation data:", X_valid.shape)

Training data: (342280, 20)
Validation data: (85571, 20)


In [27]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_valid)

In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

rf_mae = mean_absolute_error(y_valid, rf_pred)
rf_mse = mean_squared_error(y_valid, rf_pred)
rf_rmse = np.sqrt(rf_mse)

print("Random Forest Performance")
print("MAE:", rf_mae)
print("MSE:", rf_mse)
print("RMSE:", rf_rmse)

Random Forest Performance
MAE: 74.86850885561648
MSE: 24736.85719397103
RMSE: 157.27955109921643


In [29]:
from sklearn.metrics import r2_score

rf_r2 = r2_score(y_valid, rf_pred)

print("Random Forest R2 Score:", rf_r2)

Random Forest R2 Score: 0.8297725746412059


In [30]:
from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.03,
    depth=10,
    loss_function='RMSE',
    random_seed=42,
    verbose=100
)

cat_model.fit(X_train, y_train)

0:	learn: 372.1747845	total: 240ms	remaining: 4m 47s
100:	learn: 162.2817461	total: 5.69s	remaining: 1m 1s
200:	learn: 147.2120985	total: 11.5s	remaining: 57.2s
300:	learn: 138.6074179	total: 17.4s	remaining: 52s
400:	learn: 131.6786648	total: 23.9s	remaining: 47.7s
500:	learn: 126.0475466	total: 30.6s	remaining: 42.7s
600:	learn: 121.5009816	total: 37.4s	remaining: 37.3s
700:	learn: 117.8855262	total: 44s	remaining: 31.3s
800:	learn: 115.0434232	total: 50.9s	remaining: 25.3s
900:	learn: 112.1943734	total: 57.8s	remaining: 19.2s
1000:	learn: 109.7885454	total: 1m 6s	remaining: 13.2s
1100:	learn: 107.8136331	total: 1m 16s	remaining: 6.92s
1199:	learn: 105.8639389	total: 1m 26s	remaining: 0us


CatBoostRegressor(depth=10, iterations=1200, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=100)

In [31]:
cat_pred = cat_model.predict(X_valid)

In [32]:
cat_mae = mean_absolute_error(y_valid, cat_pred)
cat_mse = mean_squared_error(y_valid, cat_pred)
cat_rmse = np.sqrt(cat_mse)
cat_r2 = r2_score(y_valid, cat_pred)

print("CatBoost Performance")
print("MAE:", cat_mae)
print("MSE:", cat_mse)
print("RMSE:", cat_rmse)
print("R2 Score:", cat_r2)

CatBoost Performance
MAE: 67.25966299021692
MSE: 17341.045560931125
RMSE: 131.6854037504959
R2 Score: 0.8806670744096665


In [33]:
# Week related features
train["week_of_year"] = train["week"] % 52
train["month"] = (train["week"] % 52) // 4
train["quarter"] = train["week"] // 13

In [34]:
train["price_diff"] = train["base_price"] - train["checkout_price"]

In [35]:
train["promo_combo"] = train["emailer_for_promotion"] + train["homepage_featured"]

In [36]:

lgb_model = LGBMRegressor(
    n_estimators=4000,
    learning_rate=0.01,
    num_leaves=220,
    max_depth=15,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

lgb_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012555 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2666
[LightGBM] [Info] Number of data points in the train set: 342280, number of used features: 20
[LightGBM] [Info] Start training from score 261.568181
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

LGBMRegressor(colsample_bytree=0.9, learning_rate=0.01, max_depth=15,
              n_estimators=4000, num_leaves=220, random_state=42,
              subsample=0.9)

In [37]:
lgb_pred = lgb_model.predict(X_valid)

In [38]:
lgb_mae = mean_absolute_error(y_valid, lgb_pred)
lgb_mse = mean_squared_error(y_valid, lgb_pred)
lgb_rmse = np.sqrt(lgb_mse)
lgb_r2 = r2_score(y_valid, lgb_pred)

print("LightGBM Performance")
print("MAE:", lgb_mae)
print("MSE:", lgb_mse)
print("RMSE:", lgb_rmse)
print("R2 Score:", lgb_r2)

LightGBM Performance
MAE: 62.17669430071087
MSE: 15562.514036606124
RMSE: 124.74980575778916
R2 Score: 0.8929060924842458


In [39]:
import pickle, os, json
 
os.makedirs('../backend/models', exist_ok=True)
 
pickle.dump(rf_model,  open('../backend/models/rf_model.pkl',   'wb'))
pickle.dump(cat_model, open('../backend/models/cat_model.pkl',  'wb'))
pickle.dump(lgb_model, open('../backend/models/lgbm_model.pkl', 'wb'))
 
json.dump(list(X_train.columns), open('../backend/models/feature_cols.json', 'w'))
 
print("Saved: rf_model.pkl, cat_model.pkl, lgbm_model.pkl, feature_cols.json")
print("Feature columns:", list(X_train.columns))

Saved: rf_model.pkl, cat_model.pkl, lgbm_model.pkl, feature_cols.json
Feature columns: ['week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'category', 'cuisine', 'city_code', 'region_code', 'center_type', 'op_area', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'rolling_mean_4', 'rolling_mean_8', 'discount']
